# 04 Outcome in the reports of 2025 and 2026

For the units that were call-only up to the end of 2024, T30 and T32, the 13 UBS reports of 2025 and 2026 were searched for the call figure and read for the outcome: achieved, missed, revised, open or no longer mentioned.

In [1]:
# project setup - run first, do not edit except NAME
NAME = "call_only_targets"

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}")

ok  python 3.13.15  pandas 2.2.3


In [2]:
# folders and helpers shared by the notebooks of this section
import json, re
NB_DIR = ROOT / "notebooks" / "05_call_only_targets"
sys.path.insert(0, str(NB_DIR))
import report_checks as rc

IN = DATA / "call_only_targets"        # delivered result files
REPORTS = DATA / "reports"             # report files, optional (see 00_get_reports)
EXPORTS = NB_DIR / "exports"
EXPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160, "display.max_rows", 60)

manifest = pd.read_csv(IN / "reports_manifest.csv", keep_default_na=False)
status = rc.report_status(REPORTS, manifest)
print(rc.mode_message(status))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.


## Does a call figure appear as a target?

In [3]:
units = pd.read_csv(IN / "outcomes_units.csv", keep_default_na=False)
classes = pd.read_csv(IN / "classes_2025_2026.csv", keep_default_na=False)
classes.pivot_table(index="unit", columns="report", values="class", aggfunc="first")

report,2025 annual report,2025-Q1,2025-Q1 Pillar 3,2025-Q2,2025-Q2 Pillar 3,2025-Q3,2025-Q3 Pillar 3,2025-Q4,2025-Q4 Pillar 3,2026-Q1,2026-Q1 Pillar 3,2026-Q2,2026-Q2 Pillar 3
unit,,,,,,,,,,,,,
F11,C,C,C,C,C,C,C,C,C,C,C,C,C
F12,C,C,C,C,C,C,C,C,C,C,C,C,C
F22,C,C,C,C,C,C,C,C,C,C,C,C,C
T16,B,B,C,B,B,B,C,B,C,C,C,C,C
T25,B,B,C,B,B,B,C,B,C,C,C,C,C
T27,B,B,C,B,B,B,C,B,C,C,C,C,C
T30,C,C,C,C,C,C,C,B,C,B,C,B,C
T32,B,B,C,B,C,B,B,B,B,B,C,B,C
T33,C,C,C,C,C,C,C,C,C,C,C,C,C


No call figure appears as a target in the 13 reports (no class A). The reports set targets of their own instead: the 2024-Q4 report replaces the NCL targets of the call with RWA of around USD 29bn by the end of 2025 and around USD 22bn by the end of 2026.

In [4]:
forms_read = pd.read_csv(IN / "equivalent_forms_read.csv", keep_default_na=False)
new_targets = forms_read[forms_read.text.str.contains("around USD 29bn by the end of 2025")]
new_targets.drop_duplicates("file")[["file", "page"]].assign(quote="aim to reduce Non-core and Legacy RWA to around USD 29bn by the end of 2025 and around USD 22bn by the end of 2026")

,file,page,quote
659,UBS_2024-Q4_report.htm,,aim to reduce Non-core and Legacy RWA to around USD 29bn by the end of 2025 and around USD 22bn by the end of 2026
669,UBS_2024_annual_report.htm,,aim to reduce Non-core and Legacy RWA to around USD 29bn by the end of 2025 and around USD 22bn by the end of 2026


## Outcome per unit

In [5]:
outcome = units[["unit", "deadline", "final_status", "actual_value", "source", "page_or_table"]]
outcome.to_csv(EXPORTS / "outcomes_2025_2026.csv", index=False)
outcome

,unit,deadline,final_status,actual_value,source,page_or_table
0,T16,2026-12-31,open,NCL operational risk RWA USD 24.0bn at every reporting date from 31.3.2025 to 30.6.2026 (USD 27.1bn at 31.12.2024); call figure around USD 14bn by end-2026,UBS_2025-Q1_report.htm; UBS_2025-Q2_report.htm; UBS_2025-Q3_report.htm; UBS_2025-Q4_report.htm; UBS_2025_annual_report.htm; UBS_2026-Q1_report.htm; UBS_2026...,"table 'Composition of Non-core and Legacy', row 'Operational risk', column RWA; pages 31, 35, 37, 31, 107, 32, 35"
1,T25,2026-12-31,achieved,over USD 6bn of capital released by end-2024 (2024 annual report); USD 8bn freed up since inception by end-2025 (2025 annual report); call figure over USD 6...,UBS_2024_annual_report.htm; UBS_2025_annual_report.htm,2024 annual report pages 26 and 224; 2025 annual report page 26
2,T27,2024-12-31,achieved,NCL RWA other than operational risk USD 14.3bn at 31.12.2024 (total 41.4 less operational risk 27.1; calculated); credit and market risk RWA around USD 5bn ...,UBS_2024-Q4_report.htm; UBS_2024_annual_report.htm,"table 'Composition of Non-core and Legacy', rows 'Operational risk' and 'Total', column RWA; pages 32 and 108"
3,T30,2026-12-31,revised,cumulative integration-related expenses USD 14.2bn at 30.6.2026 (constant FX),UBS_2026-Q2_report.htm,page 7
4,T32,2030-01-01,revised,"fully applied RWA add-ons 1.44% + 1.08% = 2.52% against 1.44% before: an increase of about 108 basis points (calculated), against around 180 in the call; on...",UBS_2023_annual_report.htm; UBS_2024_annual_report.htm; UBS_2024-Q1_report.htm; UBS_2025-Q3_report.htm; UBS_2025_annual_report.htm; UBS_2026-Q2_report.htm,pages 186; 162; 42; 47; 157; 44
5,T33,2024-12-31,missed,new AT1 issuances of USD 3.5bn in 2024 (call: up to USD 2bn),UBS_2024_annual_report.htm,page 165
6,T68,2025-12-31,no longer mentioned,HoldCo not reported under this name,,
7,F11,2030-01-01,no longer mentioned,going concern capital ratio 19.0% at 30.6.2026 (18.2% at 31.3.2025),UBS_2026-Q2_report.htm,page 46
8,F12,2026-12-31,open,"Group LRD, cumulative change from asset size and other movements since 31.12.2023, currency effects excluded, in USD bn (calculated): 31.12.2024: -102.3; 31...",UBS_2024_annual_report.htm; UBS_2025_annual_report.htm; UBS_2026-Q1_report.htm; UBS_2026-Q2_report.htm,pages 169 (also 96); 93; 46; 50
9,F22,going forward,achieved,quarterly average LCR 177.3% in 2Q26 (all quarters 2025 and 2026 between 177.3% and 182.6%),UBS_2026-Q2_report.htm,page 52


In [6]:
units.groupby("final_status").unit.agg(lambda x: ", ".join(sorted(x))).to_frame("units")

,units
final_status,
achieved,"F22, T25, T27"
missed,T33
no longer mentioned,"F11, T68"
open,"F12, T16"
revised,"T30, T32"


Three commitments are achieved, two revised, one missed, two open and two no longer mentioned. No report links an outcome to the commitment of the call.

## Key values, recalculated

In [7]:
q = pd.read_csv(IN / "quotes_all.csv")
def quote_of(unit, fragment):
    hit = q[(q.origin == "outcomes_units.csv") & (q.unit == unit) & q.quote.str.contains(fragment, regex=False)]
    assert len(hit) >= 1, (unit, fragment)
    return hit.iloc[0]
def usd(text, pattern):
    return float(re.search(pattern, text).group(1))

# F12: movement of the Group LRD from asset size and other movements, currency effects excluded, in USD bn
m2024 = -usd(quote_of("F12", "During 2024").quote, r"asset size and other movements of USD ([\d.]+)bn")
m2025 = -usd(quote_of("F12", "During 2025").quote, r"USD ([\d.]+)bn decrease from asset size")
m2026q1 = usd(quote_of("F12", "USD 1,653.5bn").quote, r"USD ([\d.]+)bn increase from asset size")
m2026q2 = usd(quote_of("F12", "USD 1,649.8bn").quote, r"USD ([\d.]+)bn increase from asset size")
f12 = pd.Series({"31.12.2024": m2024, "31.12.2025": m2024 + m2025, "31.3.2026": m2024 + m2025 + m2026q1,
                 "30.6.2026": m2024 + m2025 + m2026q1 + m2026q2}).round(1)
f12.to_frame("cumulative change of the LRD since 31.12.2023, USD bn")

,"cumulative change of the LRD since 31.12.2023, USD bn"
31.12.2024,-102.3
31.12.2025,-138.4
31.3.2026,-97.8
30.6.2026,-92.2


The call figure of F12, a reduction of the Group LRD by over 100bn at constant FX within three years, was reached at the end of 2024 and of 2025 and stood at 92.2bn at mid-2026. The values exclude currency effects and the separately reported Basel III change of USD 28.8bn.

In [8]:
# T32: increase of the RWA add-ons when fully applied, in basis points
new = quote_of("T32", "on a fully applied basis").quote
old = quote_of("T32", "resulting in add-ons of").quote
fully = [float(x) for x in re.findall(r"\(([\d.]+)% on a fully applied basis\)", new)]
before = usd(old, r"add-ons of ([\d.]+)% of RWA")
t32_bp = round((sum(fully) - before) * 100)
req_2025 = usd(quote_of("T32", "14.99%").quote, r"are ([\d.]+)% of RWA")
print("fully applied add-ons:", fully, "| before:", before, "| increase:", t32_bp, "basis points")
print("requirement of", req_2025, "% plus the increase:", round(req_2025 + t32_bp / 100, 2), "% (call: 16.7%)")

fully applied add-ons: [1.44, 1.08] | before: 1.44 | increase: 108 basis points
requirement of 14.99 % plus the increase: 16.07 % (call: 16.7%)


For T32 the fully applied add-ons of 1.44% and 1.08% replace add-ons of 1.44% in total, an increase of 108 basis points against around 180 in the call. The reports give the add-ons and the USD effect, not the sum in basis points.

In [9]:
# T16 and T27: table "Composition of Non-core and Legacy", read from the reports when they are present
table_files = ["UBS_2024-Q4_report.htm", "UBS_2025-Q1_report.htm", "UBS_2025-Q2_report.htm", "UBS_2025-Q3_report.htm",
               "UBS_2025-Q4_report.htm", "UBS_2026-Q1_report.htm", "UBS_2026-Q2_report.htm"]
if all((REPORTS / f).exists() for f in table_files):
    rows = []
    for f in table_files:
        t = rc.ncl_table(REPORTS / f)
        rows.append({"file": f, "date": t["dates"][0], "operational_risk_rwa": t["operational_risk"][0],
                     "total_rwa": t["total_rwa"][0]})
    ncl = pd.DataFrame(rows)
    ncl["other_rwa"] = (ncl.total_rwa - ncl.operational_risk_rwa).round(1)
    t27_value, t16_values = ncl.other_rwa[0], set(ncl.operational_risk_rwa[1:])
    display(ncl)
else:
    print("reports not present: table values taken from the delivered outcome table")
    u = units.set_index("unit")
    t27_value = usd(u.actual_value["T27"], r"USD ([\d.]+)bn at 31\.12\.2024")
    t16_values = {usd(u.actual_value["T16"], r"USD ([\d.]+)bn at every reporting date")}
print("T27, NCL RWA other than operational risk at 31.12.2024:", t27_value, "| T16, operational risk RWA 2025 to 2026:", t16_values)

,file,date,operational_risk_rwa,total_rwa,other_rwa
0,UBS_2024-Q4_report.htm,31.12.24,27.1,41.4,14.3
1,UBS_2025-Q1_report.htm,31.3.25,24.0,34.2,10.2
2,UBS_2025-Q2_report.htm,30.6.25,24.0,32.7,8.7
3,UBS_2025-Q3_report.htm,30.9.25,24.0,30.7,6.7
4,UBS_2025-Q4_report.htm,31.12.25,24.0,28.8,4.8
5,UBS_2026-Q1_report.htm,31.3.26,24.0,28.0,4.0
6,UBS_2026-Q2_report.htm,30.6.26,24.0,27.7,3.7


T27, NCL RWA other than operational risk at 31.12.2024: 14.3 | T16, operational risk RWA 2025 to 2026: {24.0}


NCL RWA other than operational risk stood at USD 14.3bn at the end of 2024, substantially below the 40bn of the call (T27). NCL operational risk RWA stayed at USD 24.0bn at every reporting date from March 2025 to June 2026, against a call figure of around 14bn by the end of 2026 (T16).

In [10]:
t25 = quote_of("T25", "freed up").quote
t25_bn = usd(t25, r"freed up USD ([\d.]+) billion")
print(t25)

Since its inception in 2023, NCL has freed up USD 8 billion of capital


## Verification of quotes

In [11]:
later = q[q.file.str.contains("2025|2026") | (q.origin == "outcomes_units.csv")]
check = rc.verify_quotes(later, REPORTS)
check.to_csv(EXPORTS / "quote_check_2025_2026.csv", index=False)
print(rc.mode_message(status))
print("quotes:", len(check), "| checked:", int(check.checked.sum()), "| verified:", int(check.verified.sum()))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.
quotes: 79 | checked: 79 | verified: 79


In [12]:
# checks against the primary values
assert len(units) == 10 and not classes["class"].eq("A").any()
assert units.groupby("final_status").unit.agg(set).to_dict() == {
    "achieved": {"T25", "T27", "F22"}, "revised": {"T30", "T32"}, "missed": {"T33"},
    "open": {"T16", "F12"}, "no longer mentioned": {"T68", "F11"}}
assert f12.to_dict() == {"31.12.2024": -102.3, "31.12.2025": -138.4, "31.3.2026": -97.8, "30.6.2026": -92.2}
assert t32_bp == 108 and fully == [1.44, 1.08]
assert t27_value == 14.3 and t16_values == {24.0}
assert t25_bn == 8.0
assert check[check.checked].verified.all(), "a quote was not found on its page"
print("all checks passed")

all checks passed
